## 1. Perkenalan Dataset

Dataset yang digunakan dalam eksperimen ini adalah **Heart Disease Dataset** dalam file `heart.csv`.

Dataset terdiri dari **1.025 baris data dan 14 kolom**, yang terdiri dari **13 fitur prediktor** dan **1 kolom target**. Kolom `target` digunakan sebagai variabel yang akan diprediksi dalam permasalahan klasifikasi biner.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'heart.csv' in files:
        print(os.path.join(root, 'heart.csv'))

/content/drive/MyDrive/MSML_Muhammad Ausid Addari/heart.csv


## 2. Import Library

Pada tahap ini dilakukan import library yang diperlukan untuk proses eksperimen machine learning. Library yang digunakan meliputi Pandas dan NumPy untuk pengolahan data, Matplotlib dan Seaborn untuk visualisasi, serta Scikit-learn untuk preprocessing, pembagian dataset, dan pembangunan model.

In [16]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 3. Memuat Dataset

Dataset yang digunakan dalam eksperimen ini adalah dataset Heart Disease yang disimpan dalam format CSV. Dataset dimuat dari Google Drive untuk digunakan pada tahapan eksplorasi data dan preprocessing.

In [17]:
DATA_PATH = '/content/drive/MyDrive/MSML_Muhammad Ausid Addari/heart.csv'

df = pd.read_csv(DATA_PATH)

print('Dataset berhasil dimuat.')
print('Jumlah baris  :', df.shape[0])
print('Jumlah kolom  :', df.shape[1])

Dataset berhasil dimuat.
Jumlah baris  : 1025
Jumlah kolom  : 14


In [18]:
display(df.head())

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


## 4. Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) dilakukan untuk memahami karakteristik dataset sebelum menentukan tahapan preprocessing. Analisis mencakup struktur data, tipe data, missing values, duplikasi, statistik deskriptif, dan distribusi variabel target.

In [23]:
print('--- Informasi Dataset ---')
df.info()

print('--- Missing Values ---')
print(df.isnull().sum())

print('\nTotal missing values:', df.isnull().sum().sum())

print('--- Jumlah Duplikasi ---')
print('Baris duplikat:', df.duplicated().sum())

display(df.describe().T)

--- Informasi Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB
--- Missing Values ---
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0

,count,mean,std,min,25%,50%,75%,max
age,1025.0,54.434146,9.072290,29.0,48.0,56.0,61.0,77.0
sex,1025.0,0.695610,0.460373,0.0,0.0,1.0,1.0,1.0
cp,1025.0,0.942439,1.029641,0.0,0.0,1.0,2.0,3.0
trestbps,1025.0,131.611707,17.516718,94.0,120.0,130.0,140.0,200.0
chol,1025.0,246.000000,51.592510,126.0,211.0,240.0,275.0,564.0
fbs,1025.0,0.149268,0.356527,0.0,0.0,0.0,0.0,1.0
restecg,1025.0,0.529756,0.527878,0.0,0.0,1.0,1.0,2.0
thalach,1025.0,149.114146,23.005724,71.0,132.0,152.0,166.0,202.0
exang,1025.0,0.336585,0.472772,0.0,0.0,0.0,1.0,1.0
oldpeak,1025.0,1.071512,1.175053,0.0,0.0,0.8,1.8,6.2


In [24]:
print('--- Distribusi Target ---')

target_counts = df['target'].value_counts().sort_index()

print(target_counts)

print('\nProporsi target (%):')

target_percentage = (
    df['target']
    .value_counts(normalize=True)
    .sort_index() * 100
).round(2)

print(target_percentage)

--- Distribusi Target ---
target
0    499
1    526
Name: count, dtype: int64

Proporsi target (%):
target
0    48.68
1    51.32
Name: proportion, dtype: float64


## 5. Data Preprocessing

Berdasarkan hasil Exploratory Data Analysis (EDA), ditemukan 723 baris duplikat dan tidak ditemukan missing values. Oleh karena itu, preprocessing dilakukan dengan menghapus data duplikat, memisahkan fitur dan target, membagi dataset menjadi data latih dan data uji secara stratified, serta melakukan standardisasi fitur.

Tahapan preprocessing ini menjadi dasar untuk implementasi otomatisasi preprocessing pada file `pre-processing.py`.

In [26]:
df_clean = df.drop_duplicates().copy()

print('Jumlah baris awal       :', len(df))
print('Duplikasi               :', df.duplicated().sum())
print('Total missing values    :', df_clean.isnull().sum().sum())
print('Jumlah baris setelah cleaning:', len(df_clean))

retained_percentage = (len(df_clean) / len(df)) * 100
print('Baris yang dipertahankan:', round(retained_percentage, 2), '%')

Jumlah baris awal       : 1025
Duplikasi               : 723
Total missing values    : 0
Jumlah baris setelah cleaning: 302
Baris yang dipertahankan: 29.46 %


In [27]:
X = df_clean.drop(columns=['target'])
y = df_clean['target']

print('Ukuran fitur (X):', X.shape)
print('Ukuran target (y):', y.shape)

Ukuran fitur (X): (302, 13)
Ukuran target (y): (302,)


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)

print('\nProporsi target data latih:')
print(y_train.value_counts(normalize=True).sort_index().round(3))

print('\nProporsi target data uji:')
print(y_test.value_counts(normalize=True).sort_index().round(3))

X_train: (241, 13)
X_test : (61, 13)
y_train: (241,)
y_test : (61,)

Proporsi target data latih:
target
0    0.456
1    0.544
Name: proportion, dtype: float64

Proporsi target data uji:
target
0    0.459
1    0.541
Name: proportion, dtype: float64


In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Ukuran X_train_scaled:', X_train_scaled.shape)
print('Ukuran X_test_scaled :', X_test_scaled.shape)

Ukuran X_train_scaled: (241, 13)
Ukuran X_test_scaled : (61, 13)


## 6. Keputusan Preprocessing

Berdasarkan hasil Exploratory Data Analysis dan eksperimen preprocessing, ditetapkan beberapa keputusan preprocessing untuk digunakan pada proses otomatisasi.

1. **Penghapusan duplikasi**  
   Dataset awal memiliki 1.025 baris dan ditemukan 723 baris duplikat. Setelah penghapusan duplikasi, diperoleh 302 baris data. Oleh karena itu, penghapusan duplikasi menggunakan `drop_duplicates()` digunakan sebagai bagian dari preprocessing.

2. **Missing values**  
   Hasil pemeriksaan menunjukkan bahwa seluruh kolom memiliki 0 missing values. Oleh karena itu, tidak diperlukan imputasi atau penghapusan baris berdasarkan missing values pada dataset yang digunakan dalam eksperimen.

3. **Pemisahan fitur dan target**  
   Kolom `target` digunakan sebagai variabel target, sedangkan 13 kolom lainnya digunakan sebagai fitur prediktor.

4. **Pembagian dataset**  
   Dataset dibagi menjadi data latih dan data uji dengan rasio 80:20. Pembagian menggunakan `stratify=y` untuk mempertahankan proporsi kelas target dan `random_state=42` agar hasil pembagian dapat direproduksi. Hasil pembagian adalah 241 data latih dan 61 data uji.

5. **Standardisasi fitur**  
   Standardisasi dilakukan menggunakan `StandardScaler`. Scaler di-fit hanya pada data latih dan kemudian digunakan untuk mentransformasi data latih dan data uji. Dengan demikian, informasi dari data uji tidak digunakan dalam proses fitting scaler.

Keputusan preprocessing tersebut menjadi dasar implementasi otomatisasi pada file `pre-processing.py`.

### Validasi Otomatisasi Preprocessing

In [30]:
import importlib.util

FILE_PATH = '/content/drive/MyDrive/MSML_Muhammad Ausid Addari/pre-processing.py'

spec = importlib.util.spec_from_file_location(
    "preprocessing",
    FILE_PATH
)

preprocessing = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preprocessing)

print("pre-processing.py berhasil dimuat.")

pre-processing.py berhasil dimuat.


In [31]:
DATA_PATH = '/content/drive/MyDrive/MSML_Muhammad Ausid Addari/heart.csv'

X_train_auto, X_test_auto, y_train_auto, y_test_auto = (
    preprocessing.clean_and_preprocess_data(DATA_PATH)
)

In [32]:
print('--- Hasil Preprocessing Otomatis ---')
print('X_train:', X_train_auto.shape)
print('X_test :', X_test_auto.shape)
print('y_train:', y_train_auto.shape)
print('y_test :', y_test_auto.shape)

--- Hasil Preprocessing Otomatis ---
X_train: (241, 13)
X_test : (61, 13)
y_train: (241,)
y_test : (61,)
